# Fabric Client — Concurrent Iteration

Fetch every workspace's datasets, dataflows, tables, and queries in a
single fully-parallel ``TaskGroup`` with error collection.

## Setup & Authentication

Load credentials from environment variables and build the client via
the dependency-injection container.  Log level is set to ``INFO`` to
show operational highlights (scan progress, scoped fetches, timings).

In [ ]:
from collections import defaultdict

import pandas as pd
from dotenv import load_dotenv

from fabric_client import Credentials, FabricContainer, scan

# Load credentials from .env (FABRIC_CLI_TENANT_ID, FABRIC_CLI_CLIENT_ID,
# FABRIC_CLI_CLIENT_SECRET)
load_dotenv()

creds = Credentials.env()

container = FabricContainer()
container.credentials.override(creds)
client = container.client()
client.set_log_level("DEBUG")

def display(d: defaultdict, name: str) -> pd.DataFrame:
    """Show a collected data category as a DataFrame."""
    return pd.DataFrame(d[name])

In [5]:
concurrent_data

defaultdict(list,
            {'workspaces': [{'id': '131cc2c7-a36c-4973-9827-8bebefdeaea5',
               'display_name': 'Sample',
               'description': '',
               'capacity_id': '7577ac7b-79c7-4528-a575-196858a64003',
               'capacity_region': 'East Asia',
               'domain_id': None,
               'type': 'Workspace'},
              {'id': '666dfafb-ff0c-495e-982b-d98e5eef4d95',
               'display_name': '华星名仕',
               'description': '',
               'capacity_id': '7577ac7b-79c7-4528-a575-196858a64003',
               'capacity_region': 'East Asia',
               'domain_id': '8aee752a-eb55-42db-833f-383dc999f20c',
               'type': 'Workspace'}],
             'datasets': [{'id': '6436aa89-200b-4f0f-9157-54fe9e352316',
               'name': 'SampleSecondary',
               'configured_by': 'guangjun.liang@databi.cc',
               'web_url': 'https://app.powerbi.com/groups/131cc2c7-a36c-4973-9827-8bebefdeaea5/datasets/6436aa89

In [12]:
datasets = await client.datasets.list(group_id='666dfafb-ff0c-495e-982b-d98e5eef4d95')
datasets

2026-07-06 06:23:37.759 [DEBUG] fabric_client.core.http_session  GET https://api.powerbi.com/v1.0/myorg/groups/666dfafb-ff0c-495e-982b-d98e5eef4d95/datasets | params={} body={}
2026-07-06 06:23:37.873 [DEBUG] fabric_client.core.http_session  GET https://api.powerbi.com/v1.0/myorg/groups/666dfafb-ff0c-495e-982b-d98e5eef4d95/datasets → 200 (113ms)


[{'id': '86c472bd-d0ae-436c-999b-ef312abc2184',
  'name': '一汽日报',
  'webUrl': 'https://app.powerbi.com/groups/666dfafb-ff0c-495e-982b-d98e5eef4d95/datasets/86c472bd-d0ae-436c-999b-ef312abc2184',
  'addRowsAPIEnabled': False,
  'configuredBy': 'guangjun.liang@databi.cc',
  'isRefreshable': True,
  'isEffectiveIdentityRequired': False,
  'isEffectiveIdentityRolesRequired': False,
  'isOnPremGatewayRequired': False,
  'targetStorageMode': 'PremiumFiles',
  'createdDate': '2026-06-04T12:26:32.423Z',
  'createReportEmbedURL': 'https://app.powerbi.com/reportEmbed?config=eyJjbHVzdGVyVXJsIjoiaHR0cHM6Ly9XQUJJLUVBU1QtQVNJQS1CLVBSSU1BUlktcmVkaXJlY3QuYW5hbHlzaXMud2luZG93cy5uZXQiLCJlbWJlZEZlYXR1cmVzIjp7InVzYWdlTWV0cmljc1ZOZXh0Ijp0cnVlfX0%3d',
  'qnaEmbedURL': 'https://app.powerbi.com/qnaEmbed?config=eyJjbHVzdGVyVXJsIjoiaHR0cHM6Ly9XQUJJLUVBU1QtQVNJQS1CLVBSSU1BUlktcmVkaXJlY3QuYW5hbHlzaXMud2luZG93cy5uZXQiLCJlbWJlZEZlYXR1cmVzIjp7InVzYWdlTWV0cmljc1ZOZXh0Ijp0cnVlfX0%3d',
  'upstreamDatasets': [],
  'user

## Concurrent Iteration (all workspaces at once)

Fan out every workspace's datasets and dataflows in a single
``TaskGroup``, then tables and queries for each child resource.

In [ ]:
# Fully parallel across all workspaces — single TaskGroup.
# Errors are collected so one failure never stops the rest.
import asyncio
from typing import Any

concurrent_data = defaultdict(list)
errors: list[dict[str, Any]] = []

def _collect_error(resource_type: str, resource_id: str, exc: BaseException) -> None:
    errors.append({
        "type": resource_type,
        "id": resource_id,
        "error": f"{type(exc).__name__}: {exc}",
    })

async def _gather_dataset_safe(ds):
    try:
        tables, queries, parameters = await asyncio.gather(
            ds.tables, ds.queries, ds.parameters,
        )
    except Exception as exc:
        _collect_error("dataset", str(ds.id), exc)
        return ds, [], [], []
    return ds, tables, queries, parameters

async def _gather_dataflow_safe(df):
    try:
        queries = await df.queries
    except Exception as exc:
        _collect_error("dataflow", str(df.id), exc)
        return df, []
    return df, queries

async def _gather_workspace(ws):
    """Fetch scoped resources for a workspace, then fan out child fetches."""
    datasets, dataflows = await asyncio.gather(ws.datasets, ws.dataflows)
    ds_coros = [_gather_dataset_safe(ds) for ds in datasets]
    df_coros = [_gather_dataflow_safe(df) for df in dataflows]
    child_results = await asyncio.gather(*ds_coros, *df_coros)
    return ws, datasets, dataflows, child_results, len(datasets)

_filtered = [w for w in await client.workspaces if w.display_name is not None]
workspaces = [ws async for ws in scan(_filtered)]

async with asyncio.TaskGroup() as tg:
    tasks = [tg.create_task(_gather_workspace(ws)) for ws in workspaces]

for task in tasks:
    ws, datasets, dataflows, child_results, n_ds = task.result()
    concurrent_data["workspaces"].append(ws.model_dump())

    ds_results = child_results[:n_ds]
    df_results = child_results[n_ds:]

    for ds, tables, queries, parameters in ds_results:
        concurrent_data["datasets"].append(ds.model_dump())
        for t in tables:
            concurrent_data["tables"].append(t.model_dump())
        for q in queries:
            concurrent_data["queries"].append(q.model_dump())
        for p in parameters:
            concurrent_data["parameters"].append(p.model_dump())

    for df, queries in df_results:
        concurrent_data["dataflows"].append(df.model_dump())
        for q in queries:
            concurrent_data["queries"].append(q.model_dump())

print(
    f"workspaces={len(concurrent_data['workspaces'])}, "
    f"datasets={len(concurrent_data['datasets'])}, "
    f"dataflows={len(concurrent_data['dataflows'])}, "
    f"tables={len(concurrent_data['tables'])}, "
    f"parameters={len(concurrent_data['parameters'])}, "
    f"queries={len(concurrent_data['queries'])}"
)

if errors:
    client._logger.warning("%d error(s) collected:", len(errors))
    for err in errors:
        client._logger.warning("  [%s] %s — %s", err["type"], err["id"], err["error"])
else:
    client._logger.info("No errors collected.")

## View Results

Use the ``display()`` helper to inspect collected data as DataFrames.

In [ ]:
display(concurrent_data,"parameters")